# Retrieval-Augmented Generation (RAG) with Python + NVIDIA API

**Environment:** Google Colab  
**LLM:** NVIDIA NIM through LangChain  
**Vector store:** FAISS  
**Supported documents:** PDF, TXT, DOCX, CSV, XLSX

This notebook builds a practical RAG pipeline: load documents → clean and split text → create embeddings → index chunks → retrieve relevant context → generate an answer with citations to source files.

> Before running: add your NVIDIA API key to **Google Colab → Secrets** with the name `NVIDIA_API_KEY`, and enable notebook access to that secret. Never paste an API key into a shared notebook.


## 1. Install dependencies

In [ ]:
%pip install -qU langchain langchain-community langchain-text-splitters langchain-nvidia-ai-endpoints faiss-cpu pypdf pandas openpyxl docx2txt

## 2. Configure NVIDIA API

In [ ]:
from google.colab import userdata
import os

NVIDIA_API_KEY = userdata.get("NVIDIA_API_KEY")
if not NVIDIA_API_KEY:
    raise ValueError("Add NVIDIA_API_KEY in Colab Secrets and enable notebook access.")

os.environ["NVIDIA_API_KEY"] = NVIDIA_API_KEY
print("NVIDIA API key loaded securely.")

In [ ]:
from langchain_nvidia_ai_endpoints import ChatNVIDIA, NVIDIAEmbeddings

# If a model is unavailable for your NVIDIA account, choose an enabled model
# from the NVIDIA API Catalog and replace the model name.
llm = ChatNVIDIA(
    model="meta/llama-3.3-70b-instruct",
    temperature=0.2,
    max_tokens=1024,
)

embeddings = NVIDIAEmbeddings(
    model="nvidia/nv-embedqa-e5-v5"
)

print("LLM and embedding model initialized.")

## 3. Add your documents

Upload one or more files to Colab. You can upload PDF, TXT, DOCX, CSV, or XLSX files.

In [ ]:
from google.colab import files

uploaded = files.upload()
file_paths = list(uploaded.keys())
print("Uploaded files:", file_paths)

## 4. Load PDF, TXT, DOCX, CSV and Excel files

In [ ]:
from pathlib import Path
import pandas as pd

from langchain_core.documents import Document
from langchain_community.document_loaders import PyPDFLoader, TextLoader, Docx2txtLoader

documents = []

for file_path in file_paths:
    suffix = Path(file_path).suffix.lower()

    if suffix == ".pdf":
        loaded = PyPDFLoader(file_path).load()

    elif suffix == ".txt":
        loaded = TextLoader(file_path, encoding="utf-8").load()

    elif suffix == ".docx":
        loaded = Docx2txtLoader(file_path).load()

    elif suffix == ".csv":
        df = pd.read_csv(file_path).fillna("")
        loaded = [
            Document(
                page_content="\n".join(f"{col}: {row[col]}" for col in df.columns),
                metadata={"source": file_path, "row": int(i) + 1}
            )
            for i, row in df.iterrows()
        ]

    elif suffix in [".xlsx", ".xls"]:
        sheets = pd.read_excel(file_path, sheet_name=None)
        loaded = []
        for sheet_name, df in sheets.items():
            df = df.fillna("")
            for i, row in df.iterrows():
                content = "\n".join(f"{col}: {row[col]}" for col in df.columns)
                loaded.append(Document(
                    page_content=content,
                    metadata={"source": file_path, "sheet": sheet_name, "row": int(i) + 1}
                ))

    else:
        print(f"Skipped unsupported file: {file_path}")
        continue

    # Ensure source metadata is available for every loaded document.
    for doc in loaded:
        doc.metadata.setdefault("source", file_path)

    documents.extend(loaded)

print(f"Loaded {len(documents)} document units.")
if not documents:
    raise ValueError("No supported documents were loaded. Upload PDF, TXT, DOCX, CSV, or XLSX files.")

## 5. Inspect and clean the extracted text

In [ ]:
def clean_text(text):
    # Basic whitespace cleanup; retain meaningful punctuation and line breaks.
    import re
    text = text.replace("\x00", " ")
    text = re.sub(r"[ \t]+", " ", text)
    text = re.sub(r"\n{3,}", "\n\n", text)
    return text.strip()

clean_documents = []
for doc in documents:
    content = clean_text(doc.page_content)
    if content:
        clean_documents.append(Document(
            page_content=content,
            metadata=doc.metadata.copy()
        ))

documents = clean_documents
print("Non-empty document units:", len(documents))
print("\nSample text:\n", documents[0].page_content[:1000])
print("\nSample metadata:", documents[0].metadata)

## 6. Split documents into chunks

In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=150,
    separators=["\n\n", "\n", ". ", " ", ""]
)

chunks = text_splitter.split_documents(documents)
print("Total chunks:", len(chunks))
print("\nFirst chunk:\n", chunks[0].page_content[:700])

## 7. Create embeddings and build the FAISS vector database

In [ ]:
from langchain_community.vectorstores import FAISS

vector_store = FAISS.from_documents(
    documents=chunks,
    embedding=embeddings
)

print("FAISS vector index created with", len(chunks), "chunks.")

## 8. Retrieve relevant chunks

In [ ]:
def retrieve_context(question, k=4):
    results = vector_store.similarity_search(question, k=k)
    return results

question = "Write a question about the documents you uploaded."
retrieved_docs = retrieve_context(question, k=4)

for i, doc in enumerate(retrieved_docs, start=1):
    print(f"--- Result {i} ---")
    print("Metadata:", doc.metadata)
    print(doc.page_content[:700], "\n")

## 9. Build the RAG prompt and generate an answer

In [ ]:
from langchain_core.prompts import ChatPromptTemplate

rag_prompt = ChatPromptTemplate.from_messages([
    ("system", """You are a careful question-answering assistant.
Answer using only the supplied context.
If the context does not contain the answer, say that you cannot find the answer in the uploaded documents.
Do not invent facts. Mention the source filename(s) used when possible.

Context:
{context}"""),
    ("human", "{question}")
])

def format_context(docs):
    blocks = []
    for i, doc in enumerate(docs, start=1):
        source = doc.metadata.get("source", "unknown")
        page = doc.metadata.get("page")
        location = f", page {page + 1}" if isinstance(page, int) else ""
        blocks.append(
            f"[Source {i}: {source}{location}]\n{doc.page_content}"
        )
    return "\n\n".join(blocks)

def ask_rag(question, k=4):
    retrieved = retrieve_context(question, k=k)
    context = format_context(retrieved)
    messages = rag_prompt.format_messages(
        context=context,
        question=question
    )
    response = llm.invoke(messages)

    print("ANSWER\n")
    print(response.content)
    print("\nRETRIEVED SOURCES")
    for i, doc in enumerate(retrieved, start=1):
        print(f"{i}. {doc.metadata}")
    return response, retrieved

# Replace this with a question grounded in your uploaded documents.
response, sources = ask_rag("What are the main topics discussed in these documents?")

## 10. Interactive question-answering loop

In [ ]:
while True:
    user_question = input("Ask a question (or type 'exit'): ").strip()
    if user_question.lower() in {"exit", "quit"}:
        print("Chat ended.")
        break
    if user_question:
        ask_rag(user_question, k=4)
        print("\n" + "=" * 80 + "\n")

## 11. Save and reload the FAISS index

Save the index to Google Drive if you want to reuse it in a later Colab session. Only load indexes you created or trust: FAISS persistence uses local files and LangChain's loader may deserialize metadata.

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

INDEX_DIR = "/content/drive/MyDrive/rag_faiss_index"
vector_store.save_local(INDEX_DIR)
print("Saved FAISS index to:", INDEX_DIR)

In [ ]:
# Run this in a later session after initializing `embeddings`.
# Only set allow_dangerous_deserialization=True for an index you created/trust.
loaded_vector_store = FAISS.load_local(
    INDEX_DIR,
    embeddings,
    allow_dangerous_deserialization=True
)

# Replace the active vector store with the reloaded one.
vector_store = loaded_vector_store
print("FAISS index reloaded.")

## 12. Evaluate retrieval manually

Use questions whose answers are present in your documents. Check whether the retrieved chunks contain the evidence before judging the generated answer.

In [ ]:
test_questions = [
    "What is the purpose of this document?",
    "List the important concepts explained.",
    "What conclusions or recommendations are given?"
]

for q in test_questions:
    print("\nQUESTION:", q)
    docs = retrieve_context(q, k=4)
    for i, doc in enumerate(docs, 1):
        print(f"\nRetrieved {i} | {doc.metadata}")
        print(doc.page_content[:500])

## Troubleshooting

- **401 / 403:** Verify the NVIDIA API key, account access, and enabled model.
- **Model not found:** Select a model available to your NVIDIA API account and update `ChatNVIDIA(model=...)`.
- **Embedding error:** Check that the embedding model is enabled for your key.
- **PDF extraction is empty:** The PDF may be scanned. Apply OCR (for example, Tesseract) before loading.
- **Out-of-memory:** Reduce `chunk_size`, `k`, or the number of uploaded documents.
- **Poor answers:** Inspect retrieved chunks, improve document cleaning, adjust chunk size/overlap, and test a different embedding model.

## What you built

You now have a basic multi-format RAG application with NVIDIA embeddings and chat completion, FAISS similarity search, source metadata, interactive Q&A, and index persistence.